# Phase 1 – Data Cleaning & Quality Check
**IndiaKart E-Commerce | Analyst: Vaishali Girase**

Goal: never trust raw data blindly. We profile every table, check nulls, duplicates, data types, outliers and table-to-table links (referential integrity).

In [1]:
import pandas as pd, numpy as np, glob, os
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format

# 1) Put this notebook in the SAME folder as the CSV files (or change DATA_DIR)
DATA_DIR = "/kaggle/input/datasets/vaishaligirase/indiakart"

def load(name):
    """Finds e.g. 'orders DA.csv' or 'orders.csv' automatically."""
    hits = glob.glob(os.path.join(DATA_DIR, name + "*.csv"))
    return pd.read_csv(hits[0])

orders, items, customers = load("orders"), load("order_items"), load("customers")
products, payments = load("products"), load("payments")
returns, inventory, suppliers = load("returns"), load("inventory"), load("suppliers")

# 2) Convert dates (Indian format DD-MM-YYYY)
orders["order_date"] = pd.to_datetime(orders["order_date"], format="%d-%m-%Y")
orders["delivered_date"] = pd.to_datetime(orders["delivered_date"], format="%d-%m-%Y")
customers["registration_date"] = pd.to_datetime(customers["registration_date"], format="%d-%m-%Y")
returns["return_date"] = pd.to_datetime(returns["return_date"], format="%d-%m-%Y")

## Step 1 – Profile every table (rows, columns, data types)

In [2]:
tables = {"orders":orders,"order_items":items,"customers":customers,"products":products,
          "payments":payments,"returns":returns,"inventory":inventory,"suppliers":suppliers}
profile = pd.DataFrame({n:{"rows":d.shape[0],"columns":d.shape[1]} for n,d in tables.items()}).T
profile["total_records"] = profile["rows"]
print("Total records:", profile["rows"].sum())
profile

Total records: 222200


,rows,columns,total_records
orders,50000,19,50000
order_items,100000,11,100000
customers,10000,17,10000
products,1000,17,1000
payments,50000,13,50000
returns,10000,10,10000
inventory,1000,11,1000
suppliers,200,14,200


## Step 2 – Missing values (which columns, and what % of rows)

In [3]:
rows=[]
for n,d in tables.items():
    for col,cnt in d.isna().sum().items():
        if cnt>0: rows.append([n,col,cnt,round(cnt/len(d)*100,2)])
nulls = pd.DataFrame(rows,columns=["table","column","null_count","null_%"])
nulls

,table,column,null_count,null_%
0,orders,delivered_date,17501,35.00
1,payments,refund_date,50000,100.00


**Finding:** only `orders.delivered_date` (and `payments.refund_date`, which is empty because no refunds are recorded in `payments`) have nulls. Let me confirm the orders nulls are *legitimate* (not delivered yet).

In [4]:
# Is delivered_date missing ONLY for orders that were never delivered?
print(orders.groupby("status")["delivered_date"].apply(lambda s: s.notna().sum()).rename("orders_with_delivered_date"))
print("\nPayments with refund_amount > 0:", (payments.refund_amount>0).sum())

status
Cancelled         0
Delivered     32499
Processing        0
Returned          0
Shipped           0
Name: orders_with_delivered_date, dtype: int64

Payments with refund_amount > 0: 0


**Conclusion:** Nulls in `delivered_date` are valid – only Delivered orders have a date. No cleaning needed. `payments.refund_date` is 100% empty → refunds are tracked only in `returns.csv`.

## Step 3 – Duplicates

In [5]:
for n,d in tables.items():
    print(f"{n:12s} duplicate rows: {d.duplicated().sum()}")
print("\nDuplicate order_id in orders:", orders["order_id"].duplicated().sum())
print("Duplicate customer_id in customers:", customers["customer_id"].duplicated().sum())

orders       duplicate rows: 0
order_items  duplicate rows: 0
customers    duplicate rows: 0
products     duplicate rows: 0
payments     duplicate rows: 0
returns      duplicate rows: 0
inventory    duplicate rows: 0
suppliers    duplicate rows: 0

Duplicate order_id in orders: 0
Duplicate customer_id in customers: 0


## Step 4 – Data type validation

In [6]:
print(orders[["order_date","delivered_date"]].dtypes)
print("\nAmount columns numeric? ", all(pd.api.types.is_numeric_dtype(orders[c]) for c in ["total_amount","gst_amount","discount_amount","final_amount"]))
print("Date range:", orders["order_date"].min().date(), "to", orders["order_date"].max().date())

order_date        datetime64[ns]
delivered_date    datetime64[ns]
dtype: object

Amount columns numeric?  True
Date range: 2023-06-24 to 2025-06-23


## Step 5 – Delivered date must be AFTER order date

In [7]:
bad = orders[orders["delivered_date"] < orders["order_date"]]
print("Rows where delivered_date < order_date:", len(bad))
d = (orders["delivered_date"]-orders["order_date"]).dt.days.dropna()
print("Average delivery time (days): %.1f | min %d | max %d" % (d.mean(), d.min(), d.max()))

Rows where delivered_date < order_date: 0
Average delivery time (days): 4.5 | min 2 | max 7


## Step 6 – Payment failure rate

In [8]:
vc = payments["status"].value_counts()
print(vc)
print("\nFailure rate: %.2f%%" % (vc["Failed"]/len(payments)*100))

# WHY do payments fail? Link failed payments to order status
m = payments.merge(orders[["order_id","status"]], on="order_id", suffixes=("_pay","_order"))
pd.crosstab(m["status_pay"], m["status_order"])

status
Success    48252
Failed      1748
Name: count, dtype: int64

Failure rate: 3.50%


status_order,Cancelled,Delivered,Processing,Returned,Shipped
status_pay,,,,,
Failed,1748,0,0,0,0
Success,4248,32499,2501,3933,5071


**Insight:** every one of the 1,748 failed payments belongs to a *Cancelled* order → ~29% of all cancellations are caused by payment failure.

## Step 7 – Referential integrity (do child rows have a parent?)

In [9]:
checks = {
 "returns.order_id -> orders": (~returns["order_id"].isin(orders["order_id"])).sum(),
 "order_items.order_id -> orders": (~items["order_id"].isin(orders["order_id"])).sum(),
 "orders.customer_id -> customers": (~orders["customer_id"].isin(customers["customer_id"])).sum(),
 "payments.order_id -> orders": (~payments["order_id"].isin(orders["order_id"])).sum(),
 "order_items.product_id -> products": (~items["product_id"].isin(products["product_id"])).sum(),
 "inventory.product_id -> products": (~inventory["product_id"].isin(products["product_id"])).sum(),
}
pd.Series(checks, name="orphan_rows")

returns.order_id -> orders            0
order_items.order_id -> orders        0
orders.customer_id -> customers       0
payments.order_id -> orders           0
order_items.product_id -> products    0
inventory.product_id -> products      0
Name: orphan_rows, dtype: int64

## Step 8 – Business-logic checks (these are the "hidden" quality issues)

In [10]:
# (a) Returns linked to orders whose status is NOT Delivered/Returned
r = returns.merge(orders[["order_id","status"]], on="order_id")
print(r["status"].value_counts())
print("\nReturns on Cancelled/Shipped orders (logically impossible):", r["status"].isin(["Cancelled","Shipped"]).sum())

# (b) Orders placed BEFORE the customer registered
x = orders.merge(customers[["customer_id","registration_date"]], on="customer_id")
print("\nOrders placed before customer registration date:", (x["order_date"]<x["registration_date"]).sum())

# (c) customers.total_orders vs actual orders
actual = orders.groupby("customer_id").size()
customers["actual_orders"] = customers["customer_id"].map(actual).fillna(0)
print("Customers whose total_orders != actual order count:", (customers["total_orders"]!=customers["actual_orders"]).sum())

# (d) Customers with 0 orders
print("Customers with total_orders = 0:", (customers["total_orders"]==0).sum())

# (e) final_amount equals total_amount (dictionary says shipping/discount are separate)
print("final_amount == total_amount in %.1f%% of orders" % ((orders["final_amount"]==orders["total_amount"]).mean()*100))

status
Delivered    4327
Returned     3933
Cancelled    1067
Shipped       673
Name: count, dtype: int64

Returns on Cancelled/Shipped orders (logically impossible): 1740

Orders placed before customer registration date: 10255
Customers whose total_orders != actual order count: 0
Customers with total_orders = 0: 457
final_amount == total_amount in 98.2% of orders


## Step 9 – Outliers in final_amount (> ₹5 lakh)

In [11]:
out = orders[orders["final_amount"]>500000]
print("Orders above ₹5 lakh:", len(out), "(%.2f%% of orders)" % (len(out)/len(orders)*100))
print("Their share of GMV: %.1f%%" % (out["final_amount"].sum()/orders["final_amount"].sum()*100))
print(orders["final_amount"].describe())
orders["is_high_value"] = orders["final_amount"]>500000   # flag, do NOT delete

Orders above ₹5 lakh: 312 (0.62% of orders)
Their share of GMV: 5.8%
count      50,000.00
mean       62,947.71
std        97,897.50
min           136.07
25%         6,893.52
50%        24,183.38
75%        71,495.74
max     1,133,382.48
Name: final_amount, dtype: float64


## Step 10 – Inventory check

In [12]:
print(inventory["status"].value_counts())
inventory[inventory["status"]=="Out of Stock"][["product_id","warehouse_location","quantity_available","reorder_level"]]

status
In Stock        922
Low Stock        76
Out of Stock      2
Name: count, dtype: int64


,product_id,warehouse_location,quantity_available,reorder_level
853,PRD0854,Chennai-WH5,0,42
899,PRD0900,Bengaluru-WH3,0,26


## Data Quality Report (summary – paste into your 1-page report)

| # | Check | Result | Action |
|---|-------|--------|--------|
| 1 | Total records loaded | 222,200 rows across 8 tables | – |
| 2 | Duplicate rows / duplicate order_id | 0 | None |
| 3 | Nulls | Only `delivered_date` (17,501 = 35%, valid: undelivered orders) and `refund_date` (100%) | Keep, documented |
| 4 | delivered_date before order_date | 0 | None |
| 5 | Payment failure rate | 1,748 / 50,000 = **3.50%** (target <2%) | Flag to management |
| 6 | Returns without matching order | 0 | None |
| 7 | Out-of-stock products | **2** (+76 low stock) | Flag |
| 8 | Customers with 0 orders | 457 | Noted, fine |
| 9 | Orders > ₹5 lakh | 312 | Flagged, kept |
| 10 | Returns on Cancelled/Shipped orders | 1,740 | **Logic issue – flag to data owner** |
| 11 | Orders dated before customer registered | 10,255 | **Logic issue – flag** |

**Conclusion:** the data is structurally clean (no duplicates, no orphans, valid dates). The issues found are *business-logic inconsistencies*, so they are flagged and documented rather than silently deleted.